In [2]:
!pip install pymongo

                                              0.0/873.2 kB ? eta -:--:--
     --------------------                   471.0/873.2 kB 9.8 MB/s eta 0:00:01
     -------------------------------------- 873.2/873.2 kB 9.3 MB/s eta 0:00:00
                                              0.0/307.7 kB ? eta -:--:--
     ------------------------------------- 307.7/307.7 kB 19.8 MB/s eta 0:00:00


### psycopg2 for interacting with PostgreSQL
### pymongo for interacting with MongoDBdatabase

In [1]:
import psycopg2
from pymongo import MongoClient
from config import *


In [8]:
def connect_postgres():
    conn = psycopg2.connect(
        host=PG_HOST,
        database=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD
    )
    return conn

def connect_mongo():
    client = MongoClient(MONGO_HOST)
    return client[MONGO_DATABASE]

In [9]:
pg_conn = connect_postgres()
mongo_db = connect_mongo()
cursor = pg_conn.cursor()
cursor.execute("SELECT * FROM students")
students = cursor.fetchall()

### ETL pipeline

In [12]:
def extract_data(conn):
    cursor = conn.cursor()

    # Extract students
    cursor.execute("SELECT * FROM students")
    students = cursor.fetchall()

    # Extract departments
    cursor.execute("SELECT * FROM department")
    departments = cursor.fetchall()

    # Extract courses
    cursor.execute("SELECT * FROM courses")
    courses = cursor.fetchall()

    # Extract instructors
    cursor.execute("SELECT * FROM instructors")
    instructors = cursor.fetchall()

    # Extract enrollments
    cursor.execute("SELECT * FROM enrollments")
    enrollments = cursor.fetchall()

    cursor.close()
    return students, departments, courses, instructors, enrollments

def load_data(mongo_db, student_data, department_data, instructor_data):
    # Inserting student data
    mongo_db.students.insert_many(student_data)
    
    # Inserting department data
    mongo_db.departments.insert_many(department_data)

    # Inserting instructor data
    mongo_db.instructors.insert_many(instructor_data)

def transform_data(students, departments, courses, instructors, enrollments):
    # Dictionary for departments
    department_dict = {dept[0]: {"department_name": dept[1], "courses": []} for dept in departments}

    # Dictionary for courses
    course_dict = {course[0]: {"course_name": course[1], "credits": course[2], "category": course[3], "department_id": course[4]} for course in courses}

    # Dictionary for instructors
    instructor_dict = {instructor[0]: {"instructor_name": instructor[1], "email": instructor[2], "phone": instructor[3], "department_id": instructor[4], "courses_taught": []} for instructor in instructors}

    # Student data for MongoDB
    student_data = []
    for student in students:
        student_id = student[0]
        student_info = {
            "_id": student_id,
            "student_name": student[1],
            "email": student[2],
            "dob": str(student[3]),
            "phone": student[4],
            "admission_year": student[5],
            "gender": student[6],
            "department_id": student[7],
            "enrollments": []
        }

        # Enrollments for the student
        for enrollment in enrollments:
            if enrollment[1] == student_id:
                course_id = enrollment[2]
                instructor_id = enrollment[3]
                enrolled_semester = enrollment[4]

                if course_id in course_dict and instructor_id in instructor_dict:
                    course_info = course_dict[course_id]
                    instructor_info = instructor_dict[instructor_id]

                    student_info["enrollments"].append({
                        "course_id": course_id,
                        "course_name": course_info["course_name"],
                        "credits": course_info["credits"],
                        "category": course_info["category"],
                        "instructor": {
                            "instructor_id": instructor_id,
                            "instructor_name": instructor_info["instructor_name"],
                            "email": instructor_info["email"],
                            "phone": instructor_info["phone"]
                        },
                        "enrolled_semester": enrolled_semester
                    })

                    # Instructor courses_taught
                    instructor_info["courses_taught"].append({
                        "course_id": course_id,
                        "course_name": course_info["course_name"],
                        "credits": course_info["credits"],
                        "category": course_info["category"]
                    })

        student_data.append(student_info)

    # Department data for MongoDB
    department_data = []
    for dept_id, dept_info in department_dict.items():
        # Adding courses to department
        for course_id, course in course_dict.items():
            if course["department_id"] == dept_id:
                dept_info["courses"].append({
                    "course_id": course_id,
                    "course_name": course["course_name"],
                    "credits": course["credits"],
                    "category": course["category"]
                })
        dept_info["_id"] = dept_id
        department_data.append(dept_info)

    # Instructor data for MongoDB
    instructor_data = []
    for instructor_id, instructor_info in instructor_dict.items():
        instructor_info["_id"] = instructor_id
        instructor_data.append(instructor_info)

    return student_data, department_data, instructor_data


In [13]:

def main():
    pg_conn = connect_postgres()
    mongo_db = connect_mongo()

    try:
        students, departments, courses, instructors, enrollments = extract_data(pg_conn)
        student_data, department_data, instructor_data = transform_data(students, departments, courses, instructors, enrollments)
        load_data(mongo_db, student_data, department_data, instructor_data)
        print("Data migration from PostgreSQL to MongoDB completed successfully.")
    finally:
        pg_conn.close()

if __name__ == "__main__":
    main()


Data migration from PostgreSQL to MongoDB completed successfully.
